In [1]:
# LIBRARY and GLOBAL IMPORTS
import os, ray, sys, re, glob, csv
sys.path.append(".")
from functionsG import *

# === Core Libraries ===
# Import necessary libraries
import numpy as np
from collections import Counter
from collections import defaultdict, deque
import pandas as pd

# === Directories ===
var_dir = './running_vars'
uic_dir = './big_dfs/2_UIC_hier_data'

# def create_df_suff_lkup(df_lkup):
#     df_lkup_unique_suff = df_lkup.copy()
#     df_lkup_unique_suff['uic_suff'] = df_lkup_unique_suff['asg_uic'].str[-2:]
#     df_lkup_unique_suff['uic_pde_suff'] = df_lkup_unique_suff['asg_uic_pde'].str[-4:]
#     df_lkup_unique_suff = df_lkup_unique_suff.drop(columns=['asg_uic_pde','asg_uic']).drop_duplicates()
#     suff_dict = df_lkup_unique_suff.uic_suff.value_counts().to_dict()
#     unique_suffs = [suff for suff in suff_dict if suff_dict[suff] == 1]
#     df_suff_lkup = df_lkup_unique_suff[df_lkup_unique_suff.uic_suff.isin(unique_suffs)]
#     return df_suff_lkup

# df_suff_lkup = create_df_suff_lkup(load_feather('df_lu_uic_1to1'))
# store_feather(df_suff_lkup,'df_suff_lkup',uic_dir)

# UIC Hierarchy Analysis Functions
def find_subordinate_uics_recursive(df_in, 
                                   top_uic, 
                                   uic_col = 'UIC', 
                                   parent_col = 'PARENTUIC',
                                   include_top = True,
                                   max_depth = None,
                                   show=False):
    """
    Recursively find all subordinate UICs for a given top-level UIC.
    
    Parameters:
    -----------
    df : pd.DataFrame
    DataFrame containing UIC hierarchy data
    top_uic : str, top-level UIC to get all subordinates for
    List of top-level UICs to start the search from
    uic_col : str, default 'UIC'
    Column name containing the unit identification codes
    parent_col : str, default 'PARENTUIC'
    Column name containing the parent/superior UIC for each unit
    include_top : bool, default True
    Whether to include the top-level UICs in the results
    max_depth : Optional[int], default None
    Maximum depth to search (None for unlimited depth)
    
    Returns:
    --------
    sub_uics : list
    list of all subordinate UICs
    """
    df=df_in.copy()
    # Validate inputs
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if uic_col not in df.columns:
        raise ValueError(f"Column '{uic_col}' not found in DataFrame")
    if parent_col not in df.columns:
        raise ValueError(f"Column '{parent_col}' not found in DataFrame")
    
    # Remove rows with null UICs or parent UICs for cleaner processing
    clean_df = df.dropna(subset=[uic_col, parent_col])
    
    # Create lookup dictionary for faster searching
    # Group by parent UIC to get all direct subordinates
    hierarchy_dict = defaultdict(set)
    
    # Build the hierarchy dictionary
    for _, row in clean_df.iterrows():
        parent = row[parent_col]
        child = row[uic_col]
        if pd.notna(parent) and pd.notna(child):
            hierarchy_dict[parent].add(child)
    
    if show:
        print(f"📊 Built hierarchy dictionary with {len(hierarchy_dict)} parent UICs")

    subordinates = set()

    if include_top:
        subordinates.add(top_uic)
    
    # Use BFS to find all subordinates
    queue = deque([(top_uic, 0)])  # (uic, depth)
    visited = set()
    
    while queue:
        # print('fy, queue',clean_df.FY.unique().tolist(),queue)
        current_uic, depth = queue.popleft()
        
        # Skip if we've already processed this UIC
        if current_uic in visited:
            continue
            
        visited.add(current_uic)

        # Check depth limit
        if max_depth is not None and depth >= max_depth:
            continue
        
        # Get direct subordinates
        direct_subordinates = hierarchy_dict.get(current_uic, set())
        
        # Add them to df_uic_hierarchy.sub_class.unique()results and queue
        for sub_uic in direct_subordinates:
            subordinates.add(sub_uic)
            queue.append((sub_uic, depth + 1))
    
    if show:
        fy_var = None
        if 'FY' in clean_df.columns:
            fy_var ='FY'
        elif 'fy' in clean_df.columns:
            fy_var = 'fy'
        if fy_var:
            print(f"🎯 Found {len(subordinates)} total subordinates for {top_uic} for FY {clean_df[fy_var].unique().tolist()}")
        else:
            print("!!!!!!!!!!  No recognizable Fiscal Year Column found in DataFrame")
    return subordinates

In [2]:
# Get the uic_subordinate_dict_raw
print("Loading 'uic_subordinate_dict_raw'... ")
uic_subordinate_dict_raw = load_json('uic_subordinate_dict_raw', var_dir)


Loading 'uic_subordinate_dict_raw'... 


In [3]:
# === FAST PRECOMPUTE: parent -> children map per FY ===
# This avoids re-scanning the full FY dataframe for every top-level UIC.

# Load df_htables if it is not already in memory
if 'df_htables' not in locals():
    # Load the concatenated DataFrame from the hierarchy tables for XXXXXX from 2015-2016
    df_htables = load_feather('./2_UIC_hier_data/df_WDARFF_2015-2026')

# Detect FY column name in the hierarchy file
fy_col = 'FY' if 'FY' in df_htables.columns else 'fy'

# Standard column names used in WDARFF hierarchy files
uic_col = 'UIC'
parent_col = 'PARENTUIC'

# Validate required columns
missing_cols = [c for c in [fy_col, uic_col, parent_col] if c not in df_htables.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in df_htables: {missing_cols}")

# Build parent -> children dictionary once per FY
fy_parent_children = {}
for fyr in sorted(df_htables[fy_col].dropna().unique().tolist()):
    df_fy = df_htables[df_htables[fy_col] == fyr].dropna(subset=[uic_col, parent_col])
    parent_children = df_fy.groupby(parent_col)[uic_col].apply(list).to_dict()
    fy_parent_children[fyr] = parent_children

print(f"Built parent->children maps for {len(fy_parent_children):,} FYs")

 ./2_UIC_hier_data/df_WDARFF_2015-2026 Loaded!!  - (00.11 seconds and 30.705 MB of memory)
Built parent->children maps for 12 FYs


In [4]:
# === FAST BUILD: fy_uic_dict using precomputed parent->children maps ===
# This uses BFS on the in-memory dicts, which is much faster than repeated
# DataFrame filtering for every (FY, UIC) pair.

from collections import deque

def get_all_subordinates(parent_children, root_uic, max_depth=None):
    """Return all subordinate UICs for a root UIC using BFS on a dict."""
    seen = set()
    queue = deque([(root_uic, 0)])

    while queue:
        node, depth = queue.popleft()
        if node in seen:
            continue
        seen.add(node)

        # Optional depth guard (None = no depth limit)
        if max_depth is not None and depth >= max_depth:
            continue

        for child in parent_children.get(node, []):
            queue.append((child, depth + 1))

    # Remove the root itself for subordinate-only output
    seen.discard(root_uic)
    return list(seen)

build_fy_uic_dict = True

if build_fy_uic_dict:
    fy_uic_dict = {}
    count = 0
    total = len(uic_subordinate_dict_raw) * len(fy_parent_children)

    for uic in list(uic_subordinate_dict_raw.keys()):
        fy_uic_dict[uic] = {
            'top_name': uic_subordinate_dict_raw[uic]['top_name'],
            'class': uic_subordinate_dict_raw[uic]['class'],
            'sub_class': uic_subordinate_dict_raw[uic]['sub_class'],
            'fy': {}
        }

        for fyr, parent_children in fy_parent_children.items():
            sub_uics = get_all_subordinates(parent_children, uic, max_depth=None)
            count += 1
            # print(
            #     f"Computing {count} of {total} - UIC ==> {uic} "
            #     f"({uic_subordinate_dict_raw[uic]['top_name']}), "
            #     f"For FY {fyr} has {len(sub_uics)} "
            # )
            fy_uic_dict[uic]['fy'][fyr] = sorted(set(sub_uics))

    store_json(fy_uic_dict, 'fy_uic_dict', var_dir)
else:
    fy_uic_dict = load_json('fy_uic_dict', var_dir)

Saving ./running_vars/fy_uic_dict


In [5]:
# === FINAL: explode fy_uic_dict into a merge-ready DataFrame ===
# Output columns are designed for clean merges into 520 snapshots.
# Each subordinate UIC gets mapped to its top-level unit for that FY.

include_top_uic = True  # set False if you only want subordinates, not the top itself

rows = []
for top_uic, meta in fy_uic_dict.items():
    top_name = meta.get('top_name')
    top_class = meta.get('class')
    top_sub_class = meta.get('sub_class')

    for fyr, sub_uics in meta.get('fy', {}).items():
        # Optionally include the top UIC in the mapping
        if include_top_uic:
            rows.append({
                'uic': top_uic,
                'top_uic': top_uic,
                'top_name': top_name,
                'class': top_class,
                'sub_class': top_sub_class,
                'fy': fyr,
            })

        for uic in sub_uics:
            rows.append({
                'uic': uic,
                'top_uic': top_uic,
                'top_name': top_name,
                'class': top_class,
                'sub_class': top_sub_class,
                'fy': fyr,
            })

# Create the lookup DataFrame
df_uic_div_lookup = pd.DataFrame(rows)

# Division logic: only specific classes get a div_name
DIV_CLASSES = {'HDIV', 'LDIV', 'XDIV', 'SFAC', 'RGR', 'SOAR', '1SFC'}
df_uic_div_lookup['div_name'] = np.where(
    df_uic_div_lookup['class'].isin(DIV_CLASSES),
    df_uic_div_lookup['top_name'],
    np.nan,
)

# Build a by-FY USASOC UIC list (for prestige_unit in pipeline)
USASOC_TOP = 'XX'
usasoc_by_fy = (
    df_uic_div_lookup.loc[df_uic_div_lookup['top_uic'] == USASOC_TOP]
    .groupby('fy')['uic']
    .apply(lambda s: sorted(set(s.dropna())))
    .to_dict()
)
store_json(usasoc_by_fy, 'usasoc_uics_by_fy', var_dir)
print(f"Saved usasoc_uics_by_fy for {len(usasoc_by_fy):,} FYs")

# Move div_name column before top_uic
df_uic_div_lookup = df_uic_div_lookup[['uic', 'div_name', 'top_uic', 'top_name', 'class', 'sub_class', 'fy']]

# Drop duplicates to ensure one row per (uic, fy)
df_uic_div_lookup = df_uic_div_lookup.drop_duplicates(subset=['uic', 'fy'])

# Optional: save for reuse
store_feather(df_uic_div_lookup, 'df_uic_div_lookup')
print(f"Saved df_uic_div_lookup: {len(df_uic_div_lookup):,} rows")

Saving ./running_vars/usasoc_uics_by_fy
Saved usasoc_uics_by_fy for 12 FYs
 df_uic_div_lookup Stored!!  - (00.06 seconds and 3.627 MB of memory)
Saved df_uic_div_lookup: 59,424 rows


In [6]:
# df_ow = load_feather('df_oer_enriched')
# df_pw = load_feather('df_pipeline_01_base')

# snap_pid_set = set(df_pw.pid_pde.unique().tolist())
# oer_pid_set = set(df_ow.rated_ofcr.unique().tolist())
# no_oer_data_set = snap_pid_set - oer_pid_set
# df_no_oer = df_pw[df_pw.pid_pde.isin(no_oer_data_set)].copy()
# df_no_oer_cpt = df_no_oer[df_no_oer.rank_pde == 'CPT'].sort_values(by=['pid_pde','snpsht_dt'])
# df_no_oer_cpt_job = df_no_oer_cpt.drop_duplicates(subset=['pid_pde'], keep='last')
# print(f" the snap_df has {len(snap_pid_set):,} unique officers")
# print(f" the oer_df has {len(oer_pid_set):,} unique officers")

# print(f" There are {len(no_oer_data_set):,} officers in the snap dataframe not in the oer dataframe")
# print(f" Of those officers, only {df_no_oer_cpt.pid_pde.nunique():,} are Captains.")
# print(f" Their branch distribution is as follows: {df_no_oer_cpt_job.occ_crer_grp_cd.value_counts()}")

In [7]:
# === Build PDE-keyed division lookup for 520/525 merges ===
# Goal: translate clear UIC-based hierarchy output to asg_uic_pde key space
df_div = load_feather('df_uic_div_lookup')  # from this notebook
df_map = load_feather('df_lu_uic_1to1')  # clear <-> pde crosswalk
# Keep only the mapping columns we need
df_map2 = (
    df_map[['asg_uic_pde', 'asg_uic']]
        .dropna(subset=['asg_uic_pde', 'asg_uic'])
        .drop_duplicates()
)
# Map clear UIC ('uic') to PDE UIC ('asg_uic_pde')
df_uic_div_lookup_pde = (
    df_div.merge(df_map2, left_on='uic', right_on='asg_uic', how='left')
    .drop(columns=['asg_uic'])
)
# Keep useful columns and dedupe to one row per (asg_uic_pde, fy)
keep_cols = [
    'asg_uic_pde', 'fy', 'div_name',
    'top_uic', 'top_name', 'class', 'sub_class', 'uic'
]
df_uic_div_lookup_pde = df_uic_div_lookup_pde[[c for c in keep_cols if c in df_uic_div_lookup_pde.columns]]
df_uic_div_lookup_pde = df_uic_div_lookup_pde.drop_duplicates(subset=['asg_uic_pde', 'fy'])
# Save for py_503/520 use
store_feather(df_uic_div_lookup_pde, 'df_uic_div_lookup_pde')
# Quick diagnostics
print(f"Saved df_uic_div_lookup_pde: {len(df_uic_div_lookup_pde):,} rows")
print(f"Columns: {df_uic_div_lookup_pde.columns.tolist()}")
print(f"Missing asg_uic_pde rate: {df_uic_div_lookup_pde['asg_uic_pde'].isna().mean()}")
print(f"Missing div_name rate: {df_uic_div_lookup_pde['div_name'].isna().mean()}")
print(f"Unique (asg_uic_pde, fy): {df_uic_div_lookup_pde[['asg_uic_pde', 'fy']].drop_duplicates().shape[0]:,}")

 df_uic_div_lookup Loaded!!  - (00.01 seconds and 3.627 MB of memory)
 df_lu_uic_1to1 Loaded!!  - (00.01 seconds and 0.629 MB of memory)
 df_uic_div_lookup_pde Stored!!  - (00.05 seconds and 3.243 MB of memory)
Saved df_uic_div_lookup_pde: 47,224 rows
Columns: ['asg_uic_pde', 'fy', 'div_name', 'top_uic', 'top_name', 'class', 'sub_class', 'uic']
Missing asg_uic_pde rate: 0.00025410808063696424
Missing div_name rate: 0.5077079451126546
Unique (asg_uic_pde, fy): 47,224


In [19]:
df_uic_div_lookup_pde[df_uic_div_lookup_pde.fy == '15'].asg_uic_pde.nunique()

3872

In [22]:
lu_set = set(df_map.asg_uic_pde.unique())

In [18]:
df_map.drop_duplicates().asg_uic_pde.nunique()

33294

In [20]:
df_raw = load_feather('df_502_base')

 df_502_base Loaded!!  - (02.88 seconds and 1,068.091 MB of memory)


In [23]:
raw_set = set(df_raw.asg_uic_pde.unique())

In [25]:
len(raw_set - lu_set)

1

In [29]:
df_map = load_feather('df_lu_uic_1to1')
lu_set = set(df_map.asg_uic_pde.unique())
print(f" There are {len(lu_set):,} unique asg_uic_pde values in df_lu_uic_1to1\n")
df_raw = load_feather('df_502_base')
raw_set = set(df_raw.asg_uic_pde.unique())
print(f" There are {len(raw_set):,} unique asg_uic_pde values in df_502_base")
print(f"\n\n There are {len(raw_set - lu_set):,} unique asg_uic_pde values in df_502_base that are not in df_lu_uic_1to1")


 df_lu_uic_1to1 Loaded!!  - (00.01 seconds and 0.629 MB of memory)
 There are 33,294 unique asg_uic_pde values in df_lu_uic_1to1

 df_502_base Loaded!!  - (01.59 seconds and 1,068.091 MB of memory)
 There are 21,445 unique asg_uic_pde values in df_502_base


 There are 1 unique asg_uic_pde values in df_502_base that are not in df_lu_uic_1to1
